In [1]:
import pandas as pd
data = pd.read_csv('/content/drive/MyDrive/shashank_paper_2/Bhojpuri/merged_data.csv')
data.head()

,label,text
0,1,हमार सबसे बड़ समस्या बा कि सबकुछ के ओवरथिंक कईल
1,1,सबसे खराब उदासी उ उदासी ह जवन रउआ अपना के छिप...
2,1,हमरा त पहिलहीं से मालूम बा कि हम काफ़ी बढ़िया...
3,1,"अब हम उहे नइखीं। हम मान लेब, बहुत गंदगी हमरा ..."
4,1,"हमरा दिमाग में एतना कुछ बा, लेकिन अंत में हम ..."


In [2]:
text = data['text']
label = data['label']

In [3]:
from sklearn.model_selection import train_test_split
train, test, labels_train, labels_test = train_test_split(text, label, test_size=0.25, random_state=42)

In [4]:
import numpy as np
from tensorflow.keras.utils import to_categorical
#classes_list = ["ham","spam"]
#label_index = class_label.apply(classes_list.index)
label_train = to_categorical(np.asarray(labels_train))
label_test = to_categorical(np.asarray(labels_test))

In [5]:
from keras.preprocessing.text import Tokenizer
tk=Tokenizer(filters='!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n',lower=True, split=" ")
tk.fit_on_texts(train)
index=tk.word_index
#print(index)
x = tk.texts_to_sequences(train)
x1 = tk.texts_to_sequences(test)
#print (x)
vocab_size = len(index)
word_index = tk.word_index

In [6]:
from keras.preprocessing.sequence import pad_sequences
embedding_vecor_length =32
padded_docs_train = pad_sequences(x, maxlen=embedding_vecor_length, padding='post')
padded_docs_test = pad_sequences(x1, maxlen=embedding_vecor_length, padding='post')


In [7]:
#load pre-trained embedding:
embeddings_index = {}
f = open('/content/drive/My Drive/LSTM_Attention_on_sarcasm_dataset/glove.twitter.27B/glove.twitter.27B.100d.txt', encoding="utf8")
for line in f:
    values = line.split()#This splits each values in a  line with spaces
    word = values[0] # This store the first values which is the word in english
    coefs = np.asarray(values[1:], dtype='float32') # This converts all splitted values from 1 to last(expect first word) into an array
    embeddings_index[word] = coefs # english word = array of numerical values
f.close()
print('Total %s word vectors in Glove 6B 100d.' % len(embeddings_index))

Total 1193514 word vectors in Glove 6B 100d.


In [8]:
# Initialize the weight for each word for the embedding layer. Words not present
# in the dictionary initialized by random value
EMBEDDING_DIM = 100
embedding_matrix = np.random.random((len(word_index) + 1, EMBEDDING_DIM))
#embedding_matrix = (16433, 100) NOTE: It is from uniques words in our tweets
for word, i in word_index.items(): # Selecting word from our tweet unique words list for changing with the values from glove
    embedding_vector = embeddings_index.get(word) # Finding our tweet words in glove and taking its array of 100 values
    if embedding_vector is not None:
#words not found in embedding index will be all-zeros.
        embedding_matrix[i] = embedding_vector

In [9]:
import pandas as pd
from keras.models import Sequential
from keras.layers import Dense, Dropout, Activation, Flatten, Input, RepeatVector
from keras.layers import Embedding, LSTM, Bidirectional, SimpleRNN, GRU
from keras.models import Sequential, Model
from tensorflow.keras.layers import Permute
from keras import layers
from keras import backend as K
from keras.layers import Dense,Input,Dropout, concatenate,Concatenate
from keras.models import Sequential
from keras.layers import Dense, Dropout, Activation, Flatten, Input
from keras.layers import Embedding, LSTM, Bidirectional, SimpleRNN, GRU
from keras.models import Sequential, Model
import keras
from keras.models import Sequential
from keras.layers import Dense, Dropout, Activation, Flatten, Input
from keras.layers import Embedding, LSTM, Bidirectional, SimpleRNN, GRU
from keras.models import Sequential, Model

In [ ]:
units = 64
lstm_input= Input(shape=(embedding_vecor_length,),  dtype='int32', name='lstm_input')
x= Embedding(vocab_size+1, 100,weights=[embedding_matrix], input_length=embedding_vecor_length,trainable=False)(lstm_input)
x1=LSTM(256,return_sequences=True, dropout = 0.2, recurrent_dropout = 0.2)(x)
lstm_out= LSTM(64,return_sequences=True, dropout = 0.2, recurrent_dropout = 0.2)(x1)
attention = Dense(1, activation = 'tanh')(lstm_out)
attention = Flatten()(attention)
attention1 = Activation('softmax')(attention)
attention = RepeatVector(units)(attention1)
attention = Permute([2,1])(attention)

sent_representation = layers.Multiply()([lstm_out, attention])
sent_representation = layers.Lambda(lambda xin: K.sum(xin, axis = -2), output_shape = (units,))(sent_representation)
'''
input2 = Input(shape=(X_train_tfidf.shape[1],))
dense1 = Dense(512,  activation='relu')(input2)
dense2 = Dense(16, activation='relu')(dense1)
merged  = keras.layers.concatenate([sent_representation, dense2])
'''
main_output = Dense(2,activation='softmax', name='main_output')(sent_representation)
model_lstm = Model(inputs=lstm_input, outputs=main_output)

model_lstm.compile(loss='binary_crossentropy', optimizer='Adam', metrics=['accuracy'])
print(model_lstm.summary())

Model: "model"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
lstm_input (InputLayer)         [(None, 32)]         0                                            
__________________________________________________________________________________________________
embedding (Embedding)           (None, 32, 100)      241300      lstm_input[0][0]                 
__________________________________________________________________________________________________
lstm (LSTM)                     (None, 32, 256)      365568      embedding[0][0]                  
__________________________________________________________________________________________________
lstm_1 (LSTM)                   (None, 32, 64)       82176       lstm[0][0]                       
______________________________________________________________________________________________

In [ ]:
model_lstm.fit(padded_docs_train, label_train, validation_data = (padded_docs_test, label_test),
          epochs=5,batch_size=100, verbose=2)

Epoch 1/5
14/14 - 16s - loss: 0.4611 - accuracy: 0.8143 - val_loss: 0.4173 - val_accuracy: 0.8418
Epoch 2/5
14/14 - 7s - loss: 0.3987 - accuracy: 0.8428 - val_loss: 0.3630 - val_accuracy: 0.8418
Epoch 3/5
14/14 - 7s - loss: 0.3542 - accuracy: 0.8465 - val_loss: 0.3293 - val_accuracy: 0.8418
Epoch 4/5
14/14 - 7s - loss: 0.3002 - accuracy: 0.8713 - val_loss: 0.2923 - val_accuracy: 0.9014
Epoch 5/5
14/14 - 7s - loss: 0.2611 - accuracy: 0.8874 - val_loss: 0.2721 - val_accuracy: 0.8997


In [ ]:
predictions_test_lstm = model_lstm.predict(padded_docs_test)
predictions_testlstm1 = np.zeros_like(predictions_test_lstm)
predictions_testlstm1[np.arange(len(predictions_test_lstm)), predictions_test_lstm.argmax(1)] = 1
from sklearn.metrics import classification_report
print(classification_report(label_test,predictions_testlstm1))

              precision    recall  f1-score   support

           0       0.94      0.94      0.94       495
           1       0.69      0.67      0.68        93

   micro avg       0.90      0.90      0.90       588
   macro avg       0.81      0.81      0.81       588
weighted avg       0.90      0.90      0.90       588
 samples avg       0.90      0.90      0.90       588



In [ ]:
units = 64
lstm_input= Input(shape=(embedding_vecor_length,),  dtype='int32', name='lstm_input')
x= Embedding(vocab_size+1, 100,weights=[embedding_matrix], input_length=embedding_vecor_length,trainable=False)(lstm_input)
x1=Bidirectional(LSTM(256,return_sequences=True, dropout = 0.2, recurrent_dropout = 0.2))(x)
lstm_out= Bidirectional(LSTM(64,return_sequences=False, dropout = 0.2, recurrent_dropout = 0.2))(x1)
main_output = Dense(2,activation='softmax', name='main_output')(lstm_out)
model_lstm1 = Model(inputs=lstm_input, outputs=main_output)

model_lstm1.compile(loss='binary_crossentropy', optimizer='Adam', metrics=['accuracy'])
print(model_lstm1.summary())

Model: "model_1"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
lstm_input (InputLayer)      [(None, 32)]              0         
_________________________________________________________________
embedding_1 (Embedding)      (None, 32, 100)           241300    
_________________________________________________________________
bidirectional (Bidirectional (None, 32, 512)           731136    
_________________________________________________________________
bidirectional_1 (Bidirection (None, 128)               295424    
_________________________________________________________________
main_output (Dense)          (None, 2)                 258       
Total params: 1,268,118
Trainable params: 1,026,818
Non-trainable params: 241,300
_________________________________________________________________
None


In [ ]:
print(padded_docs_train.shape,label_train.shape,padded_docs_test.shape,label_test.shape)

(1368, 32) (1368, 2) (588, 32) (588, 2)


In [ ]:
model_lstm1.fit(padded_docs_train, label_train, validation_data = (padded_docs_test, label_test),
          epochs=5,batch_size=100, verbose=2)

Epoch 1/5
14/14 - 28s - loss: 0.4579 - accuracy: 0.8056 - val_loss: 0.4024 - val_accuracy: 0.8418
Epoch 2/5
14/14 - 18s - loss: 0.3843 - accuracy: 0.8428 - val_loss: 0.3454 - val_accuracy: 0.8418
Epoch 3/5
14/14 - 18s - loss: 0.3268 - accuracy: 0.8575 - val_loss: 0.3271 - val_accuracy: 0.8571
Epoch 4/5
14/14 - 18s - loss: 0.2674 - accuracy: 0.8896 - val_loss: 0.2494 - val_accuracy: 0.8895
Epoch 5/5
14/14 - 18s - loss: 0.2068 - accuracy: 0.9108 - val_loss: 0.2544 - val_accuracy: 0.8963


In [ ]:
predictions_test_lstm2 = model_lstm1.predict(padded_docs_test)
predictions_testlstm3 = np.zeros_like(predictions_test_lstm2)
predictions_testlstm3[np.arange(len(predictions_test_lstm2)), predictions_test_lstm2.argmax(1)] = 1
from sklearn.metrics import classification_report
print(classification_report(label_test,predictions_testlstm3))

              precision    recall  f1-score   support

           0       0.91      0.98      0.94       495
           1       0.79      0.47      0.59        93

   micro avg       0.90      0.90      0.90       588
   macro avg       0.85      0.72      0.77       588
weighted avg       0.89      0.90      0.89       588
 samples avg       0.90      0.90      0.90       588



In [10]:
units = 64
lstm_input= Input(shape=(embedding_vecor_length,),  dtype='int32', name='lstm_input')
x= Embedding(vocab_size+1, 100,weights=[embedding_matrix], input_length=embedding_vecor_length,trainable=False)(lstm_input)
x1=LSTM(256,return_sequences=True, dropout = 0.2, recurrent_dropout = 0.2)(x)
lstm_out= LSTM(64,return_sequences=False, dropout = 0.2, recurrent_dropout = 0.2)(x1)
main_output = Dense(2,activation='softmax', name='main_output')(lstm_out)
model_lstm2 = Model(inputs=lstm_input, outputs=main_output)

model_lstm2.compile(loss='binary_crossentropy', optimizer='Adam', metrics=['accuracy'])
print(model_lstm2.summary())

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm_input (InputLayer)     [(None, 32)]              0         
                                                                 
 embedding (Embedding)       (None, 32, 100)           1121000   
                                                                 
 lstm (LSTM)                 (None, 32, 256)           365568    
                                                                 
 lstm_1 (LSTM)               (None, 64)                82176     
                                                                 
 main_output (Dense)         (None, 2)                 130       
                                                                 
Total params: 1568874 (5.98 MB)
Trainable params: 447874 (1.71 MB)
Non-trainable params: 1121000 (4.28 MB)
_________________________________________________________________
None


In [11]:
model_lstm2.fit(padded_docs_train, label_train, validation_data = (padded_docs_test, label_test),
          epochs=10,batch_size=100, verbose=2)

Epoch 1/10
91/91 - 79s - loss: 0.6743 - accuracy: 0.6055 - val_loss: 0.6716 - val_accuracy: 0.6014 - 79s/epoch - 871ms/step
Epoch 2/10
91/91 - 67s - loss: 0.6013 - accuracy: 0.6765 - val_loss: 0.5503 - val_accuracy: 0.7247 - 67s/epoch - 735ms/step
Epoch 3/10
91/91 - 71s - loss: 0.5218 - accuracy: 0.7347 - val_loss: 0.5087 - val_accuracy: 0.7394 - 71s/epoch - 784ms/step
Epoch 4/10
91/91 - 63s - loss: 0.4959 - accuracy: 0.7438 - val_loss: 0.4929 - val_accuracy: 0.7497 - 63s/epoch - 695ms/step
Epoch 5/10
91/91 - 66s - loss: 0.4672 - accuracy: 0.7653 - val_loss: 0.4909 - val_accuracy: 0.7673 - 66s/epoch - 728ms/step
Epoch 6/10
91/91 - 67s - loss: 0.4454 - accuracy: 0.7806 - val_loss: 0.4576 - val_accuracy: 0.7743 - 67s/epoch - 733ms/step
Epoch 7/10
91/91 - 71s - loss: 0.4252 - accuracy: 0.7930 - val_loss: 0.4324 - val_accuracy: 0.7866 - 71s/epoch - 776ms/step
Epoch 8/10
91/91 - 67s - loss: 0.4125 - accuracy: 0.8011 - val_loss: 0.4246 - val_accuracy: 0.8019 - 67s/epoch - 741ms/step
Epoch 9/

In [12]:
predictions_test_lstm4 = model_lstm2.predict(padded_docs_test)
predictions_testlstm5 = np.zeros_like(predictions_test_lstm4)
predictions_testlstm5[np.arange(len(predictions_test_lstm4)), predictions_test_lstm4.argmax(1)] = 1
from sklearn.metrics import classification_report
print(classification_report(label_test,predictions_testlstm5))

94/94 [==============================] - 7s 68ms/step
              precision    recall  f1-score   support

           0       0.87      0.83      0.85      1811
           1       0.75      0.81      0.78      1197

   micro avg       0.82      0.82      0.82      3008
   macro avg       0.81      0.82      0.81      3008
weighted avg       0.82      0.82      0.82      3008
 samples avg       0.82      0.82      0.82      3008

